# 29 HM: Huff Model

**Atlas source reading copy · browser-network**

[Course guide](../index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Geospatial-Analyses-and-Maps/blob/7ff9306b61deeb68047e7c31cd4c2ccc3cc76d72/notebooks/29_HM_huff-model.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability.

**Provenance:** `notebooks/29_HM_huff-model.ipynb` at `7ff9306b61deeb68047e7c31cd4c2ccc3cc76d72`; original SHA-256 `3d50b4283e7d9cfaf9d7e4ff8cfb170fd41eb3f767e5b253eb1b6fa8f38edab3`. Original author material incorporated at repository owner request; no general license inferred. Third-party terms remain separate.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 29 HM: Huff Model

**Category:** Network Analysis  
**Primary dataset:** `nyc_taxi_sample.csv`  

This revised notebook is map-first: every analysis produces spatial evidence, not just tabular output. The examples use small local datasets so they run in a Pyodide kernel in JupyterLite.

![Layer thinking](../images/map_layers_concept.png)

<video controls width="720" src="../videos/layer_stack_demo.mp4">Layer-stack demonstration video</video>

**Source and licensing note:** bundled teaching shapefiles are synthetic CC0-style examples. GADM is referenced as a boundary-workflow inspiration, but GADM files are not redistributed. OpenStreetMap/Geofabrik and healthsites.io are referenced as workflow inspirations; the bundled road, town, and facility layers are synthetic teaching extracts.

In [ ]:
# Pyodide package bootstrap: install browser-side packages only if the kernel does not already have them.
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    import piplite
    await piplite.install(['pandas', 'numpy', 'matplotlib'])
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt

from IPython.display import display
import spatial_lite as sl

pd.set_option('display.max_columns', 40)

## 1. Load the data and read it as geography
Start with a quick table preview, then immediately map the administrative boundary context.

In [ ]:
df = sl.load_dataset('nyc_taxi_sample.csv')
value = 'fare_usd'
lon, lat = 'pickup_lon', 'pickup_lat'
sl.quick_summary(df)
sl.map_boundaries('Huff Model: administrative boundary context')

## 2. Map the subject layer
This map shows the teaching dataset as a spatial layer. Ask: where are high values, low values, clusters, or gaps?

In [ ]:
plot_df = df.rename(columns={lon:'lon', lat:'lat'}).copy()
sl.map_points(plot_df, color=value if value in plot_df.columns else None, title='Huff Model: mapped observations')

## 3. Add a second layer and compare
Spatial analysis becomes clearer when layers are compared: boundaries, facilities, roads, grids, or events.

In [ ]:
trips = df.copy()
trips['straight_line_km'] = sl.haversine(trips.pickup_lon,trips.pickup_lat,trips.dropoff_lon,trips.dropoff_lat)
sl.map_points(trips.rename(columns={'pickup_lon':'lon','pickup_lat':'lat'}), color='fare_usd', title='Pickup map colored by fare')
sl.map_points(trips.rename(columns={'dropoff_lon':'lon','dropoff_lat':'lat'}), color='duration_min', title='Dropoff map colored by travel time')
display(trips.groupby(['pickup_zone','dropoff_zone'])[['straight_line_km','duration_min','fare_usd']].mean().round(2).head(12))

## 4. Interactive map for exploration
The map below uses Leaflet and OpenStreetMap tiles when internet access is available. If tiles do not load, the static maps above still provide the analysis.

In [ ]:
try:
    sl.interactive_leaflet(plot_df, lon='lon', lat='lat', label=plot_df.columns[0], title='Interactive exploration map')
except Exception as exc:
    print('Interactive map skipped:', exc)

## Reflection
- What does the map reveal that the table hides?
- Which layer changes your interpretation most: boundaries, roads, facilities, or raster/grid context?
- What uncertainty might be introduced by scale, classification, or the choice of distance rule?

**Practice edit:** change `value = 'fare_usd'` to another numeric column and rerun the maps.